# Neural Network Basics


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasci-w266/2025-summer-main/blob/master/assignment/a1/NNBasics.ipynb)

### Brief Review of Machine Learning

In supervised learning, parametric models are those where the model is a function of a fixed form with a number of unknown _parameters_.  Together with a loss function and a training set, an optimizer can select parameters to minimize the loss with respect to the training set.  Common optimizers include stochastic gradient descent.  It tweaks the parameters slightly to move the loss "downhill" due to a small batch of examples from the training set.

## Part A:  Linear & Logistic Regression

You've likely seen linear regression before.  In linear regression, we fit a line (technically, hyperplane) that predicts a target variable, $y$, based on some features $x$.  The form of this model is affine (even if we call it "linear"):  

$$y_{hat} = xW + b$$

where $W$ and $b$ are weights and an offset, respectively, and are the parameters of this parametric model.  The loss function that the optimizer uses to fit these parameters is the squared error ($||\cdots||_2$) between the prediction and the ground truth in the training set.

You've also likely seen logistic regression, which is tightly related to linear regression.  Logistic regression also fits a line - this time separating the positive and negative examples of a binary classifier.  The form of this model is similar:

$$y_{hat} = \sigma(xW + b)$$

where again $W$ and $b$ are the parameters of this model, and $\sigma$ is the [sigmoid function](https://en.wikipedia.org/wiki/Sigmoid_function) which maps un-normalized scores ("logits") to values $\hat{y} \in [0,1]$ that represent probabilities. The loss function that the optimizer uses to fit these parameters is the [cross entropy](../../materials/lesson_notebook/lesson_1_NN_Review.ipynb) between the prediction and the ground truth in the training set.

This pattern of an affine transform, $xW + b$, occurs over and over in machine learning.

**We'll use logistic regression as our running example for the rest of this part.**


### Short Answer Questions

Imagine you want to implement logistic regression:

* `z = xW + b`
* `y_hat = sigmoid(z)`

Where:
1.  `x` is an 11-dimensional feature vector
2.  `W` is the weight vector
3.  `b` is the bias term

What are the dimensions of `W` and `b`?  Recall that in logistic regression, `z` is just a scalar (commonly referred to as the "logit").

Sketch a picture of the whole equation using rectangles to illustrate the dimensions of `x`, `W`, and `b`.  See examples below for inspiration (though please label each dimension).  We don't ask you to submit this, but make sure you can do it!  It's the "print" debugging statement of neural networks!  It's also useful for reading papers... if you can't draw the shapes of all the tensors, you don't (yet) know what's going on!

1. The dimension of W would be (11,), so after you multiply the 11-dimension x feature vector, you would also get a scalar that matches z.
2. The dimension of b would be (1) so that it matches z.

## Part B: Batching

Let's say we want to perform inference using your model (parameters `W` and `b`) above on multiple examples instead of just one. On modern hardware (especially GPUs), we can do this efficiently by *batching*.

To do this, we stack up the feature vectors in x like in the diagram below.  Note that changing the number of examples you run on (i.e. your batch size) *does not* affect the number of parameters in your model.  You're just running the same thing in parallel (instead of running the above one feature vector at a time at a time).

![](https://github.com/datasci-w266/2025-summer-main/blob/master/assignment/a1/batchaffine.png?raw=1)

The red (# features) and blue (batch size) lines represent dimensions that are the same.

### Short Answer Questions

If we have 11 features and running the model in parallel with 30 examples, what are the dimensions of:

1. `W` ?
2. `b` ?
3. `x` ?
4. `z` ?

_Hint:_ remember that your model parameters stay fixed!

1. W = (11,)
2. b = (30,)    (Since the b value is broadcasted; otherwise, it just has a single dimension for each example)
3. x = (30, 11)
4. z = (30,)

## Part C: Logistic Regression - NumPy Implementation

In this section, we'll implement logistic regression by hand and compute a few values to make sure we understand what's going on!

Let's say your model has the following parameters:

In [ ]:
import numpy as np

W = np.array([45,6,3,25,-1])
b = 5

If you want to run the model on the following three examples:

* [1, 2, 3, 4, 5]
* [0, 0, 0, 0, 5]
* [-3, -4, -12, -1, 1]

Construct the x matrix **such that you compute the answer all in one big batch** and compute the probability of the positive class for each.

In [ ]:
# Import sigmoid.
from scipy.special import expit as sigmoid

### YOUR CODE HERE
ex_1 = [1, 2, 3, 4, 5]
ex_2 = [0, 0, 0, 0, 5]
ex_3 = [-3, -4, -12, -1, 1]

# Grouping the three examples together using np.stack
x_matrix = np.stack((ex_1, ex_2, ex_3))
print("x_matrix shape:", x_matrix.shape)
print("x_matrix:")
print(x_matrix, "\n")

# Computing the probability of the positive class for each example
logits = np.dot(x_matrix, W) + b
print("Logits for the 3 examples:", logits, "\n")

# Getting the probability of the positive class for each:
prob_pos_class = sigmoid(logits)
print("Probability of positive class for each example:",
      np.round(prob_pos_class, 5))

### END YOUR CODE

x_matrix shape: (3, 5)
x_matrix:
[[  1   2   3   4   5]
 [  0   0   0   0   5]
 [ -3  -4 -12  -1   1]] 

Logits for the 3 examples: [ 166    0 -216] 

Probability of positive class for each example: [1.  0.5 0. ]


### Short Answer Questions

1. What is the probability of the positive class for the second (middle) example?
2. What is the cross-entropy loss in Base 2 of the second example if its label is positive?

In [ ]:
# Question 1:
print("Question 1: Probability of the positive class for second example:",
      prob_pos_class[1], "\n")

# Question 2:
positive_class = 1
ce = -np.dot(positive_class, np.log2(prob_pos_class[1]))
print("Question 2: Cross-entropy loss in Base 2 for the second example,",
      "assuming the label is positive:", ce)

Question 1: Probability of the positive class for second example: 0.5 

Question 2: Cross-entropy loss in Base 2 for the second example, assuming the label is positive: 1.0


1. 0.50000
2. 1.00000

## Part D: NumPy Feed Forward Neural Network

Let's do the same procedure for a simple feed-forward neural network.

Imagine you have a 3 layer network (hint: # of affines = # of layers. The affine is the W + b part of a layer).  Each hidden layer is size 10.  Just like before, you've already trained your model and you just want to run it forward.  For this exercise, let's say that each weight matrix is np.ones(...) and each bias term is [-1, -2, -3, ..., -n] if the bias term is $n$ long.  Compute the probability of the positive class for the three examples above, again in a single batch.

**Hint:  Draw the shapes of the matrices at each layer out on a piece of paper!  Include it with any questions you post to Ed Discussion.**

Assume your model uses a sigmoid as the nonlinearity for all layers.

In [ ]:
### YOUR CODE HERE

# Initializing each weight matrix as ones and the bias term as [-1, -2, ..., -n]
W1 = np.ones((5, 10))
W2 = np.ones((10, 10))
W3 = np.ones((10))
b = np.arange(1, 11) * (-1)
b_output = np.arange(1, 2) * (-1)
print("x_matrix start:")
print(x_matrix, "\n")

# Getting the logits and outputs after the first hidden layer
h1_logits = np.dot(x_matrix, W1) + b
h1_output = sigmoid(h1_logits)
print("After 1st hidden layer of size 10:")
print(h1_logits, "\n")
print("After 1st sigmoid activation:")
print(h1_output, "\n")

# Getting the logits and outputs after the second hidden layer
h2_logits = np.dot(h1_output, W2) + b
h2_output = sigmoid(h2_logits)
print("After 2nd hidden layer of size 10:")
print(h2_logits, "\n")
print("After 2nd sigmoid activation:")
print(h2_output, "\n")

# Getting the logits and outputs after the third hidden layer
h3_logits = np.dot(h2_output, W3) + b_output
h3_output = sigmoid(h3_logits)
print("After 3rd hidden layer of size 10:")
print(h3_logits, "\n")
print("After 3rd sigmoid activation:")
print(h3_output, "\n")

### END YOUR CODE

x_matrix start:
[[  1   2   3   4   5]
 [  0   0   0   0   5]
 [ -3  -4 -12  -1   1]] 

After 1st hidden layer of size 10:
[[ 14.  13.  12.  11.  10.   9.   8.   7.   6.   5.]
 [  4.   3.   2.   1.   0.  -1.  -2.  -3.  -4.  -5.]
 [-20. -21. -22. -23. -24. -25. -26. -27. -28. -29.]] 

After 1st sigmoid activation:
[[9.99999168e-01 9.99997740e-01 9.99993856e-01 9.99983299e-01
  9.99954602e-01 9.99876605e-01 9.99664650e-01 9.99088949e-01
  9.97527377e-01 9.93307149e-01]
 [9.82013790e-01 9.52574127e-01 8.80797078e-01 7.31058579e-01
  5.00000000e-01 2.68941421e-01 1.19202922e-01 4.74258732e-02
  1.79862100e-02 6.69285092e-03]
 [2.06115362e-09 7.58256042e-10 2.78946809e-10 1.02618796e-10
  3.77513454e-11 1.38879439e-11 5.10908903e-12 1.87952882e-12
  6.91440011e-13 2.54366565e-13]] 

After 2nd hidden layer of size 10:
[[  8.98939339   7.98939339   6.98939339   5.98939339   4.98939339
    3.98939339   2.98939339   1.98939339   0.98939339  -0.01060661]
 [  3.50669285   2.50669285   1.50669285 

### Short Answer Questions

1.  What is the probability of the third example?
2.  What is the cross-entropy loss if its label is negative?

In [ ]:
# Question 1:
print("Question 1: Probability of the positive class in the third example:",
      np.round(h3_output[2], 5), "\n")

# Question 2:
true_label = 0
ce = -np.dot((1 - true_label), np.log2(1 - h3_output[2]))
print("Question 2: Cross-entropy loss in Base 2 for the third example,",
      "assuming the label is negative:", np.round(ce, 5))

Question 1: Probability of the positive class in the third example: 0.36915 

Question 2: Cross-entropy loss in Base 2 for the third example, assuming the label is negative: 0.66463


1. 0.36915
2. 0.66463

## Part E: Softmax

Recall that softmax(z) is a vector with the same length as z, and whose components are:  $softmax(z)_i = \frac{e^{z_i}}{\Sigma_j e^{z_j}}$.

### Short Answer Questions

1. If the logits coming from the main body of the network are [4, 6, 8], what is the probability of the middle class?
2. What is the cross-entropy loss if the correct class is the last one? (i.e. corresponding to logit=8)?
3. If you had such a three-class classification problem, what would the dimensions of W and b be for the last layer of the feed forward neural network above?

In [ ]:
# Question 1: Getting the softmax probabilities
logits_part_e = np.array([4, 6, 8])
logits_softmax = np.exp(logits_part_e)
softmax_probs = np.divide(logits_softmax, np.sum(logits_softmax))
print("Softmax probabilities:", softmax_probs, "\n")
print("Question 1: Probability of the middle class:",
      np.round(softmax_probs[1], 5), "\n")

# Question 2: Cross-entropy loss if the correct class is the last one
y_true = np.array([0, 0, 1])
ce_loss = -np.dot(y_true, np.log2(softmax_probs))
print("Question 2: Cross-entropy loss if correct class is last one:",
      np.round(ce_loss, 5), "\n")

# Question 3: Dimensions of W and b
print("Question 3: Dimension of W: (10, 3), and b: (3,)")

Softmax probabilities: [0.01587624 0.11731043 0.86681333] 

Question 1: Probability of the middle class: 0.11731 

Question 2: Cross-entropy loss if correct class is last one: 0.20621 

Question 3: Dimension of W: (10, 3), and b: (3,)
